# ViSpatialQA — Thí nghiệm E1 + E2 cho phần Human

Mục tiêu: cải thiện **Human FR** (điểm yếu nhất), cùng YN/CO. Chi tiết lý do: `output/base/khao_sat_paper.md`.

- **E1: FR dạng checklist cho L.** Mỗi câu FR được hỏi thành 4 câu con: ngang (trái/phải), dọc (trên/dưới), khoảng cách (gần/xa), tiếp xúc. Mỗi câu con có suy luận và bỏ phiếu riêng, sau đó gộp nhãn lại. Cách này nhằm không bỏ sót nhãn "xa"/"gần".
  L không train trên Human, nên được chấm trên **Human train (149 câu FR) + dev (28)** để giảm nhiễu.
- **E2: tăng dữ liệu bằng câu đảo chiều khi train F2.** Thêm câu hỏi đảo (Y, X) cho FR (111 câu) và YN (29 câu), rồi train lại stage 2 từ adapter F1 → nguồn mới **F2C**.
- **So sánh cuối:** baseline / E1 / E2 / E1+E2, mỗi biến thể đều chấm từng nhánh, từng cặp, cả ba.

**Chạy độc lập:** mục 2b tự tạo dự đoán nền: F1/F2 bằng adapter tải từ HF, S bằng parser, L chạy lại Qwen3-32B. Chỉ cần 2 adapter đã có trên HF (repo public). Nếu muốn nhanh, đặt `BASE_PREDICTIONS="drive"` để dùng lại dự đoán đã sao lưu.

**Thời gian ước tính** (GPU ~96GB): dự đoán nền khoảng 45 phút (phần lớn là L); E1 khoảng 1–1.5 giờ; E2 khoảng 20 phút. Có thể tắt từng phần trong CONFIG.

## 1. CONFIG

In [ ]:
CONFIG = dict(
    REPO_URL="https://github.com/lightanddark0/VLSP-Task5.git",
    BRANCH="feat/fr-checklist-augment",
    WORK_DIR="/content/VLSP-Task5",
    DRIVE_DATA_DIR="/content/drive/MyDrive/VLSP2026/Data",
    DRIVE_BACKUP_DIR="/content/drive/MyDrive/VLSP2026/outputs",
    DRIVE_CKPT_DIR="/content/drive/MyDrive/VLSP2026/ckpt",
    HF_USER="",                                   # để trống = lấy từ token
    HF_F_AUTO_REPO="qwen3-8b-vi-spatial-lora-s1", # adapter F1 trên HF
    HF_F_HUMAN_REPO="qwen3-8b-vi-spatial-lora-s2", # adapter F2 trên HF
    # Dự đoán nền (F1, F2, S, LCOT) cho so sánh:
    #   "hf"    = tự suy luận lại: F1/F2 bằng adapter tải từ HF, S bằng parser, LCOT chạy lại Qwen3-32B (~40 phút)
    #   "drive" = dùng lại dự đoán đã sao lưu trên Drive nếu có, thiếu cái nào thì suy luận cái đó
    BASE_PREDICTIONS="hf",
    USE_WANDB=True, WANDB_PROJECT="vispatialqa-vlsp2026",
    F_MODEL="Qwen/Qwen3-8B",
    INFER_F=dict(max_model_len=1024, max_num_seqs=128, gpu_memory_utilization=0.90),
    LCOT=dict(model="Qwen/Qwen3-32B-FP8", fallback="Qwen/Qwen3-32B", max_tokens=8192, max_model_len=16384,
              max_num_seqs=64, fewshot_k=2, gpu_memory_utilization=0.90),
    USE_INDIFINITE=False, HUMAN_YN_DK="keep",
    SUBMISSION_NAMES=dict(human="human_submission.json", auto="auto_submission.json"),
    SEED=42,

    # --- E1: FR checklist cho L ---
    RUN_E1=True,
    E1=dict(n=8,                                  # số mẫu mỗi câu con (giảm 4 nếu muốn nhanh gấp đôi)
            baseline_on_train=True,               # chạy L thường trên Human train FR để so công bằng
            splits=["train", "dev", "test"]),
    # --- E2: tăng dữ liệu đảo chiều khi train stage 2 ---
    RUN_E2=True,
    E2=dict(augment="FR,YN", human_repeat=1, auto_mix=2000, epochs=3, learning_rate=5e-5,
            effective_batch=16, batch_sizes=[16, 8], max_len=1024,
            hub_repo="qwen3-8b-vi-spatial-lora-s2-conv"),  # public; "" = không đẩy HF
)

## 2. Chuẩn bị: code nhánh E1/E2, thư viện, Drive, dữ liệu, dự đoán cũ, HF/W&B

In [ ]:
import codecs, json, os, shlex, shutil, subprocess, sys, time
from pathlib import Path

C = CONFIG
WORK = Path(C["WORK_DIR"])
OUT = WORK / "outputs"
PRED = OUT / "predictions_e1e2"      # chỉ chứa dự đoán do notebook này tạo (độc lập với notebook chính)
CKPT = Path(C["DRIVE_CKPT_DIR"])
DRIVE_OUT = Path(C["DRIVE_BACKUP_DIR"])
os.environ["VLLM_USE_DEEP_GEMM"] = "0"            # FP8 JIT cần NVCC >= 12.9
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"   # sampler JIT cần CUDA >= 12.9 trên GPU SM 12.x
START = time.time()


def sh(cmd, check=True):
    """Chạy lệnh (list hoặc str) trong WORK_DIR, in output trực tiếp (giữ thanh tiến trình)."""
    if isinstance(cmd, (list, tuple)):
        cmd = " ".join(shlex.quote(str(part)) for part in cmd)
    print(f"$ {cmd}", flush=True)
    started = time.time()
    process = subprocess.Popen(cmd, shell=True, cwd=WORK if WORK.exists() else None, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    while chunk := os.read(process.stdout.fileno(), 4096):
        sys.stdout.write(decoder.decode(chunk)); sys.stdout.flush()
    code = process.wait()
    print(f"\n[exit {code}, {(time.time() - started) / 60:.1f} phút | tổng {(time.time() - START) / 60:.1f} phút]")
    if check and code:
        raise RuntimeError(f"Lệnh lỗi (exit {code}): {cmd}")
    return code


def gpu(title=""):
    print(f"--- nvidia-smi {title}")
    subprocess.run("nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv", shell=True)


def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return ""


def split_file(dataset, split):
    return {"train": f"Data/splits/{dataset}_train.json", "dev": f"Data/splits/{dataset}_dev.json",
            "test": f"Data/{dataset}_public_test.json"}[split]


def jobs(source, pairs):
    """--job arguments writing into outputs/predictions_e1e2/<source>/<dataset>_<split>.jsonl."""
    out = []
    for dataset, split in pairs:
        out += ["--job", f"{dataset},{split_file(dataset, split)},{PRED / source / f'{dataset}_{split}.jsonl'}"]
    return out


def to_drive(folder):
    if Path("/content/drive/MyDrive").exists() and (OUT / folder).exists():
        shutil.copytree(OUT / folder, DRIVE_OUT / folder, dirs_exist_ok=True)
        print(f"Đã sao lưu {OUT / folder} -> {DRIVE_OUT / folder}")


def train(cmd, output_dir, batch_sizes, effective_batch):
    """train_lora.py với batch lớn nhất vừa VRAM; resume đúng cấu hình nếu đã có train_shape.json."""
    shape_file = Path(output_dir) / "train_shape.json"
    if shape_file.exists():
        shape = json.loads(shape_file.read_text())
        return sh(cmd + ["--batch-size", shape["batch_size"], "--gradient-accumulation-steps",
                         shape["gradient_accumulation_steps"],
                         "--gradient-checkpointing" if shape["gradient_checkpointing"] else "--no-gradient-checkpointing"])
    attempts = [(b, False) for b in batch_sizes] + [(min(batch_sizes), True)]
    for index, (batch, checkpointing) in enumerate(attempts):
        print(f"▶ Thử batch {batch} x accum {max(1, effective_batch // batch)}, checkpointing {'bật' if checkpointing else 'tắt'}")
        args = ["--batch-size", batch, "--gradient-accumulation-steps", max(1, effective_batch // batch),
                "--gradient-checkpointing" if checkpointing else "--no-gradient-checkpointing"]
        if sh(cmd + args, check=index == len(attempts) - 1) == 0:
            return 0
        shape_file.unlink(missing_ok=True)
        print("⚠️ Lỗi (thường do thiếu VRAM) -> thử cấu hình nhẹ hơn.")


# Drive, code, thư viện
if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
if (WORK / ".git").exists():
    for args in (["fetch", "--quiet", "origin", C["BRANCH"]], ["checkout", "--quiet", "-B", C["BRANCH"], f"origin/{C['BRANCH']}"]):
        subprocess.run(["git", "-C", str(WORK), *args], check=True)
else:
    subprocess.run(["git", "clone", "--quiet", "--branch", C["BRANCH"], C["REPO_URL"], str(WORK)], check=True)
os.chdir(WORK)
print("Code:", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())
if subprocess.run([sys.executable, "-c", "import vllm, peft"], capture_output=True).returncode != 0:
    sh("pip install -q vllm")
    sh("pip install -q peft datasets accelerate huggingface_hub wandb")
sh("pip uninstall -y -q torchaudio torchao", check=False)   # bản Colab không tương thích torch của vllm

# Dữ liệu + split
for name in ["human_train.json", "auto_train.json", "human_public_test.json", "auto_public_test.json"]:
    if not (WORK / "Data" / name).exists():
        shutil.copy2(Path(C["DRIVE_DATA_DIR"]) / name, WORK / "Data" / name)
if not (WORK / "Data/splits/auto_dev.json").exists():
    sh("python make_splits.py --from-manifest")

# Resume sau khi Colab ngắt: lấy lại dự đoán mà chính notebook này đã sao lưu
for path in (DRIVE_OUT / "predictions_e1e2").rglob("*.jsonl") if (DRIVE_OUT / "predictions_e1e2").exists() else []:
    target = PRED / path.relative_to(DRIVE_OUT / "predictions_e1e2")
    if not target.exists() and not path.name.endswith(".raw.jsonl"):
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, target)
PRED.mkdir(parents=True, exist_ok=True)

# HF + W&B
os.environ.setdefault("HF_TOKEN", secret("HF_TOKEN"))
if not os.environ.get("HF_USER"):
    os.environ["HF_USER"] = C.get("HF_USER") or ""
if not os.environ["HF_USER"] and os.environ.get("HF_TOKEN"):
    from huggingface_hub import HfApi
    os.environ["HF_USER"] = HfApi(token=os.environ["HF_TOKEN"]).whoami()["name"]
if C["USE_WANDB"] and not os.environ.get("WANDB_API_KEY"):
    os.environ["WANDB_API_KEY"] = secret("WANDB_API_KEY")
os.environ.setdefault("WANDB_PROJECT", C["WANDB_PROJECT"])
os.environ["WANDB_LOG_MODEL"] = "false"
if not os.environ.get("WANDB_API_KEY"):
    os.environ["WANDB_MODE"] = "disabled"

## 2b. Dự đoán nền: F1, F2 (adapter từ HF), S, L (Qwen3-32B)
Tự tạo lại, không phụ thuộc notebook chính (hoặc dùng lại bản trên Drive nếu `BASE_PREDICTIONS="drive"`).

In [ ]:
def hf_repo(name):
    return name if "/" in name else f"{os.environ['HF_USER']}/{name}"


def have(source, files):
    return all((PRED / source / f).exists() for f in files)


if C["BASE_PREDICTIONS"] == "drive":
    for folder in ("predictions_hf", "predictions"):      # mục 17 trước, rồi bản của bước 9-12
        root = DRIVE_OUT / folder
        for path in root.rglob("*.jsonl") if root.exists() else []:
            target = PRED / path.relative_to(root)
            if not target.exists() and not path.name.endswith(".raw.jsonl") and path.parent.name in ("F1", "F2", "S", "LCOT"):
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(path, target)

IF = C["INFER_F"]
infer_args = ["--max-model-len", IF["max_model_len"], "--max-num-seqs", IF["max_num_seqs"],
              "--gpu-memory-utilization", IF["gpu_memory_utilization"], "--seed", C["SEED"],
              "--cache-dir", "/content/hf_adapters"]
F1_FILES = [f"{d}_{s}.jsonl" for d in ("auto", "human") for s in ("dev", "test")]
HUMAN_FILES = ["human_dev.jsonl", "human_test.jsonl"]
assert os.environ.get("HF_USER"), "Cần HF_TOKEN (secret) hoặc CONFIG['HF_USER'] để biết repo adapter"
if not have("F1", F1_FILES):
    gpu("trước F1")
    sh(["python", "infer_f.py", "--adapter", hf_repo(C["HF_F_AUTO_REPO"]), "--source", "F1", "--run-name", "E-base-F1-from-hf",
        *jobs("F1", [("auto", "dev"), ("human", "dev"), ("auto", "test"), ("human", "test")]), *infer_args])
if not have("F2", HUMAN_FILES):
    sh(["python", "infer_f.py", "--adapter", hf_repo(C["HF_F_HUMAN_REPO"]), "--source", "F2", "--run-name", "E-base-F2-from-hf",
        *jobs("F2", [("human", "dev"), ("human", "test")]), *infer_args])
if not have("S", ["auto_dev.jsonl", "auto_test.jsonl"] + HUMAN_FILES):
    sh(["python", "solve_symbolic.py", *jobs("S", [("auto", "dev"), ("auto", "test"), ("human", "dev"), ("human", "test")]),
        "--run-name", "E-base-S"])
if not have("LCOT", HUMAN_FILES):
    L = C["LCOT"]
    base = ["python", "llm_cot.py", "--source", "LCOT", "--n", 8, "--max-tokens", L["max_tokens"],
            "--max-model-len", L["max_model_len"], "--max-num-seqs", L["max_num_seqs"], "--fewshot-k", L["fewshot_k"],
            "--gpu-memory-utilization", L["gpu_memory_utilization"], "--seed", C["SEED"], "--fallback-model", "",
            "--run-name", "E-base-LCOT", *jobs("LCOT", [("human", "dev"), ("human", "test")])]
    gpu("trước LCOT")
    if sh(base + ["--model-name", L["model"]], check=False) != 0:
        print(f"⚠️ {L['model']} không chạy được -> thử {L['fallback']}")
        sh(base + ["--model-name", L["fallback"]])
to_drive("predictions_e1e2")
print("Dự đoán nền:", sorted(str(p.relative_to(PRED)) for p in PRED.rglob("*.jsonl") if ".raw." not in p.name))

## 3. E1 — FR checklist cho L (Qwen3-32B)
3a: L thường trên Human **train** FR (để có baseline công bằng). 3b: L checklist trên train/dev/test FR.

In [ ]:
L, E1 = C["LCOT"], C["E1"]


def lcot(source, pairs, extra):
    """llm_cot.py với FP8; nếu FP8 không khởi động được thì chạy bản BF16."""
    base = ["python", "llm_cot.py", "--source", source, "--n", E1["n"], "--q-types", "FR",
            "--max-tokens", L["max_tokens"], "--max-model-len", L["max_model_len"], "--max-num-seqs", L["max_num_seqs"],
            "--fewshot-k", L["fewshot_k"], "--gpu-memory-utilization", L["gpu_memory_utilization"],
            "--seed", C["SEED"], "--fallback-model", "", *jobs(source, pairs), *extra]
    if sh(base + ["--model-name", L["model"]], check=False) != 0:
        print(f"⚠️ {L['model']} không chạy được -> thử {L['fallback']}")
        sh(base + ["--model-name", L["fallback"]])


if C["RUN_E1"]:
    gpu("trước E1")
    if E1["baseline_on_train"]:
        lcot("LCOT", [("human", "train")], ["--fr-mode", "joint", "--run-name", "E1-LCOT-joint-human-train-FR"])
    lcot("LCOTFR", [("human", s) for s in E1["splits"]], ["--fr-mode", "checklist", "--run-name", "E1-LCOT-checklist-FR"])
    gpu("sau E1")
    to_drive("predictions_e1e2")

### 3c. So sánh FR: L thường vs L checklist (train + dev), rồi ghép FR checklist vào L → nguồn `LCOTX`

In [ ]:
from spartqa.data import read_json
from spartqa.inference import fallback_answers, score_predictions
from spartqa.postprocess import PostprocessOptions
from spartqa.predictions import read_predictions

OPTIONS = PostprocessOptions(use_indifinite=C["USE_INDIFINITE"], human_yn_dk=C["HUMAN_YN_DK"])
FALLBACK = fallback_answers(Path("Data/splits"), "human")


def fr_scores(source, split):
    path = PRED / source / f"human_{split}.jsonl"
    if not path.exists():
        return None
    records = {k: v for k, v in read_predictions(path).items() if v["q_type"] == "FR"}
    report = score_predictions(read_json(split_file("human", split)), records, "human", OPTIONS, FALLBACK)
    return report["by_task"].get("FR") if report else None


rows = ["| Tập | Nguồn | Số câu FR | Exact match | Jaccard |", "|---|---|---|---|---|"]
for split in ("train", "dev"):
    for source, label in (("LCOT", "L thường"), ("LCOTFR", "L checklist (E1)"), ("F2", "F2"), ("F1", "F1")):
        score = fr_scores(source, split)
        if score:
            rows.append(f"| {split} | {label} | {int(score['count'])} | {score['exact_match']:.3f} | {score['jaccard']:.3f} |")
from IPython.display import Markdown, display
display(Markdown("\n".join(rows)))
(OUT / "e1e2").mkdir(parents=True, exist_ok=True)
(OUT / "e1e2/e1_fr.md").write_text("\n".join(rows) + "\n", encoding="utf-8")

for split in ("dev", "test"):
    if (PRED / "LCOTFR" / f"human_{split}.jsonl").exists():
        sh(["python", "merge_predictions.py", "--base", PRED / "LCOT" / f"human_{split}.jsonl",
            "--override", PRED / "LCOTFR" / f"human_{split}.jsonl", "--q-types", "FR", "--source", "LCOTX",
            "--output", PRED / "LCOTX" / f"human_{split}.jsonl"])
to_drive("predictions_e1e2")

## 4. E2 — train lại stage 2 với câu đảo chiều → F2C

In [ ]:
E2 = C["E2"]
f1_local = CKPT / "f_auto/adapter"
F1_ADAPTER = str(f1_local) if (f1_local / "adapter_config.json").exists() else (
    C["HF_F_AUTO_REPO"] if "/" in C["HF_F_AUTO_REPO"] else f"{os.environ['HF_USER']}/{C['HF_F_AUTO_REPO']}")
F2C_DIR = CKPT / "f_human_conv"
if C["RUN_E2"]:
    sh(["python", "prepare_sft.py", "--stage", "human", "--human-repeat", E2["human_repeat"], "--auto-mix", E2["auto_mix"],
        "--augment-converse", E2["augment"], "--seed", C["SEED"], "--output-dir", OUT / "sft/human_conv"])
    print(json.loads((OUT / "sft/human_conv/stats.json").read_text())["converse_records"], "câu đảo chiều được thêm")
    if (F2C_DIR / "adapter/adapter_config.json").exists():
        print("Adapter F2C đã có trên Drive, bỏ qua train.")
    else:
        gpu("trước train F2C")
        cmd = ["python", "train_lora.py", "--train-file", OUT / "sft/human_conv/train.jsonl",
               "--dev-file", OUT / "sft/human_conv/dev.jsonl", "--model-name", C["F_MODEL"], "--init-adapter", F1_ADAPTER,
               "--output-dir", F2C_DIR, "--stage", "stage2-converse", "--dataset-tag", "human",
               "--epochs", E2["epochs"], "--learning-rate", E2["learning_rate"], "--save-strategy", "epoch",
               "--max-len", E2["max_len"], "--seed", C["SEED"], "--run-name", "E2-F2C-train"]
        if E2["hub_repo"] and os.environ.get("HF_TOKEN"):
            cmd += ["--hub-repo", E2["hub_repo"] if "/" in E2["hub_repo"] else f"{os.environ['HF_USER']}/{E2['hub_repo']}"]
        train(cmd, F2C_DIR, E2["batch_sizes"], E2["effective_batch"])
        gpu("sau train F2C")
    IF = C["INFER_F"]
    sh(["python", "infer_f.py", "--adapter", F2C_DIR / "adapter", "--source", "F2C", "--run-name", "E2-F2C-infer",
        *jobs("F2C", [("human", "dev"), ("human", "test")]), "--max-model-len", IF["max_model_len"],
        "--max-num-seqs", IF["max_num_seqs"], "--gpu-memory-utilization", IF["gpu_memory_utilization"], "--seed", C["SEED"]])
    to_drive("predictions_e1e2")

## 5. So sánh cuối: baseline / E1 / E2 / E1+E2 (mỗi biến thể: từng nhánh, từng cặp, cả ba)

In [ ]:
VARIANTS = {
    "baseline": "F=F2,F1 S=S L=LCOT",
    "E1": "F=F2,F1 S=S L=LCOTX",
    "E2": "F=F2C,F1 S=S L=LCOT",
    "E1+E2": "F=F2C,F1 S=S L=LCOTX",
    "tất cả nguồn": "F=F2C,F2,F1 S=S L=LCOTX,LCOT",
}
names = ",".join(f"{d}={n}" for d, n in C["SUBMISSION_NAMES"].items())
summary = ["| Biến thể | Tổ hợp tốt nhất | Human cv | Auto cv | **Final cv** | Human YN | Human FR | Human FB | Human CO |",
           "|---|---|---|---|---|---|---|---|---|"]
for variant, branches in VARIANTS.items():
    needed = {s for part in branches.split() for s in part.split("=")[1].split(",")}
    if not all((PRED / s / "human_dev.jsonl").exists() for s in needed if s != "S"):
        summary.append(f"| {variant} | (thiếu dự đoán: bỏ qua) | | | | | | | |")
        continue
    out_dir = OUT / "e1e2" / variant.replace(" ", "_").replace("+", "_")
    sh(["python", "compare_branches.py", "--pred-dir", PRED, "--output-dir", out_dir, "--branches", branches,
        "--human-yn-dk", C["HUMAN_YN_DK"], "--use-indifinite" if C["USE_INDIFINITE"] else "--no-use-indifinite",
        "--submission-names", names, "--run-name", f"compare-{variant.replace(' ', '_')}"])
    best = json.loads((out_dir / "best.json").read_text())
    result = json.loads((out_dir / "comparison.json").read_text())[best["combination"]]
    human, auto = result.get("human"), result.get("auto")
    fmt = lambda v: f"{v:.3f}" if v is not None else "n/a"
    per = human["cv"] if human else {}
    summary.append(f"| {variant} | {best['combination']} | {fmt(human and human['cv_mean'])} | {fmt(auto and auto['cv_mean'])} | "
                   f"**{best['final_cv']:.4f}** | " + " | ".join(
                       fmt(per.get(t, {}).get("accuracy" if t in ("YN", "CO") else "exact_match")) for t in ("YN", "FR", "FB", "CO")) + " |")
text = "\n".join(summary)
display(Markdown(text))
(OUT / "e1e2/summary.md").write_text(text + "\n\nFile nộp của từng biến thể: outputs/e1e2/<biến thể>/<tổ hợp>/\n", encoding="utf-8")
to_drive("e1e2")
print(f"Xong sau {(time.time() - START) / 60:.1f} phút. Bảng: {OUT / 'e1e2/summary.md'}")